lysosomal polygenic risk score analysis in GP2 Neurobooster genotyping data (all ancestries)

Project: GP2 lysosomal PRS

Version: Python/3.10.17, R/4.4.2

Notebook Overview

1. Description Loading Python libraries Set paths Make working directory

2. Installing packages

3. Generate genotype file in each ancestry; process clinical data; Create a covariate file with GP2 data

4. run PRSice

5. plot ROC

6. GLM analysis adjusting for sex, age, PC1-5

Getting Started

Import python dependencies

In [ ]:
## Import the necessary python dependencies 
%pip install seaborn --upgrade
!pip install rpy2
%load_ext rpy2.ipython
%pip install -U kaleido

from datetime import date
import importlib.metadata
from IPython.display import display
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, TwoSlopeNorm
from matplotlib.ticker import FormatStrFormatter
import matplotlib.gridspec as gridspec
import numbers
import numpy as np
import os
import pandas as pd
import plotly.express as px
import requests
import scipy
from scipy import stats
from scipy.stats import norm
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf
import subprocess
import sys
import seaborn as sns
import types
# Use pathlib for file path manipulation
import pathlib

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

Define helper functions

In [ ]:
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            # Split ensures you get root package, not just imported function
            name = val.__name__.split(".")[0]

        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        # Some packages are weird and have different imported names vs. system/pip names
        # Unfortunately, there is no systematic way to get pip names from a package's imported name. You'll have to add exceptions to this list manually!
        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages.keys():
            name = poorly_named_packages[name]

        yield name

def min_max_scale(data):
    return (data - np.min(data)) / (np.max(data) - np.min(data))

def compare_rocs(input_path, output_path, auc1_variable, auc2_variable, xlabel):
    # Read p-values matrix and rescale values
    df_pvals = pd.read_csv(input_path, sep="\t", index_col="Ancestry")
    df_pvals_log = np.log(-np.log(np.abs(df_pvals)) + 1) * df_pvals / np.abs(df_pvals)
    
    # Prepare formatting for heatmap
    max_abs = np.max(np.abs(df_pvals_log))
    norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
    annot = df_pvals.map(lambda x: "*" if -0.05 <= x <= 0.05 else "")
    
    # Generate heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(df_pvals_log, cmap="RdBu", norm=norm, cbar=True, ax=ax, annot=annot, fmt="")
    
    # Format color bar
    cbar = ax.collections[0].colorbar
    cbar.set_label("")
    cbar.set_ticks([-1.38522686, 1.38522686])
    cbar.set_ticklabels([f"{auc1_variable} Significantly Better", f"{auc2_variable} Significantly Better"])
    
    # Label axes and save figure
    plt.xlabel(xlabel)
    plt.ylabel("Target data ancestry")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

Print out versions of imported python dependencies

In [ ]:
imports = list(set(get_imports()))
print(f"PACKAGE VERSIONS ({date})")
for m in importlib.metadata.distributions():
    if m.metadata["Name"] in imports and m.metadata["Name"]!="pip":
        print(f"\t{m.metadata['Name']}=={m.version}")

Load R dependencies

In [ ]:
%%R

install.packages("caret")
install.packages("optparse", repos="https://cloud.r-project.org/")

In [ ]:
%%R

require(data.table)
require(dplyr)
require(ggplot2)

library(optparse)
library(data.table)
library("ggplot2")
library(RColorBrewer)
library("caret")
library("pROC")

Define directories and ancestry lists

In [ ]:
WORK_DIR = "/home/jupyter/workspace/ws_files/r11"
RESULTS_DIR = "/home/jupyter/workspace/ws_files/r11/results"
REL11_DIR = "/home/jupyter/workspace/gp2_tier2_eu_release11"

ancestries = ["AAC", "AFR", "AJ", "AMR", "SAS", "EUR", "EAS", "CAS", "MDE", "CAH"]

In [ ]:
%%R

WORK_DIR <- "/home/jupyter/workspace/ws_files/r11"
RESULTS_DIR <- "/home/jupyter/workspace/ws_files/r11/results"
REL11_DIR <- "/home/jupyter/workspace/gp2_tier2_eu_release11"

ancestries <- list("AAC", "AFR", "AJ", "AMR", "SAS", "EUR", "EAS", "CAS", "MDE", "CAH")

Install bioinformatics packages

In [ ]:
%%bash

if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter"
else
    echo "Plink is not installed"
    wget -P /home/jupyter http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
    unzip -o /home/jupyter/plink_linux_x86_64_20190304.zip -d /home/jupyter
    rm /home/jupyter/plink_linux_x86_64_20190304.zip
fi

chmod u+x /home/jupyter/plink

In [ ]:
%%bash

if test -e /home/jupyter/plink2; then
    echo "Plink2 is already installed in /home/jupyter"
else
    echo "Plink2 is not installed"
    wget -P /home/jupyter http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
    unzip -o /home/jupyter/plink2_linux_x86_64_latest.zip -d /home/jupyter
    rm /home/jupyter/plink2_linux_x86_64_latest.zip
fi

chmod u+x /home/jupyter/plink2

In [ ]:
%%bash

if test -e /home/jupyter/metal; then
    echo "Metal is already installed in /home/jupyter"
else
    echo "Metal is not installed"
    wget -P /home/jupyter https://csg.sph.umich.edu/abecasis/metal/download/Linux-metal.tar.gz
    tar --strip-components=1 -xzf /home/jupyter/Linux-metal.tar.gz -C /home/jupyter
    rm /home/jupyter/Linux-metal.tar.gz
fi

chmod u+x /home/jupyter/metal

In [ ]:
%%bash

if test -e /home/jupyter/prsice; then
    echo "PRSice is already installed in /home/jupyter"
else
    echo "PRSice is not installed"
    wget -P /home/jupyter https://github.com/choishingwan/PRSice/releases/download/2.3.5/PRSice_linux.zip
    unzip -o /home/jupyter/PRSice_linux.zip -d /home/jupyter
    mv /home/jupyter/PRSice_linux /home/jupyter/prsice
fi

chmod u+x /home/jupyter/prsice

Generate genotype file

In [ ]:
%%script bash

for ancestry in AAC AFR AJ AMR SAS; do
  for chrnum in $(seq 1 22); do
    /home/jupyter/plink2 \
      --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/${ancestry}/chr${chrnum}_${ancestry}_release11_vwb \
      --remove /home/jupyter/workspace/gp2_tier2_eu_release11/meta_data/related_samples/${ancestry}_release11_vwb.related \
      --make-bed \
      --out /home/jupyter/workspace/ws_files/r11/genotype_file/${ancestry}/chr${chrnum}
  done
done

In [ ]:
%%script bash

for ancestry in EAS CAS MDE CAH; do
  for chrnum in $(seq 1 22); do
    /home/jupyter/plink2 \
      --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/${ancestry}/chr${chrnum}_${ancestry}_release11_vwb \
      --remove /home/jupyter/workspace/gp2_tier2_eu_release11/meta_data/related_samples/${ancestry}_release11_vwb.related \
      --make-bed \
      --out /home/jupyter/workspace/ws_files/r11/genotype_file/${ancestry}/chr${chrnum}
  done
done

In [ ]:
for ancestry in EUR; do
  for chrnum in $(seq 16 22); do
    /home/jupyter/plink2 \
      --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/${ancestry}/chr${chrnum}_${ancestry}_release11_vwb \
      --remove /home/jupyter/workspace/gp2_tier2_eu_release11/meta_data/related_samples/${ancestry}_release11_vwb.related \
      --make-bed \
      --out /home/jupyter/workspace/ws_files/r11/genotype_file/${ancestry}/chr${chrnum}
  done
done

Process Clinical Data

Generate covariates with PCs

In [ ]:
CLINICAL_DATA_PATH = pathlib.Path(REL11_DIR, 'clinical_data/master_key_release11_final_vwb.csv')

In [ ]:
# Let's load the master key
key = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)
print(key.shape)
key

In [ ]:
# Extract subset where study == "TUEPAC"
FIN = key[key['nba_label'] == 'FIN']
FIN

In [ ]:
# Subsetting to keep only a few columns 
FIN = FIN[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label']]
# Renaming the columns
FIN.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'age_of_onset':'AAO'}, inplace = True)
FIN

In [ ]:
# Keep only PD and Control phenotypes (discard "Other" etc.)
valid_pheno = FIN[FIN['phenotype'].isin(['PD', 'Control'])].copy()

# Drop rows with missing AGE or SEX
valid = valid_pheno.dropna(subset=['AGE', 'SEX'])

# Count cases and controls
counts = valid['phenotype'].value_counts()
print(counts)

In [ ]:
# Extract subset where study == "TUEPAC"
tuepac = key[key['study'] == 'TUEPAC']
tuepac

In [ ]:
# Subsetting to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'age_of_onset':'AAO'}, inplace = True)
key

In [ ]:
# Reformat sex column
df_sex = key[['IID','SEX']].copy()
df_sex['SEX'] = df_sex['SEX'].replace({
    'Female':2,
    'Male':1,
    'Other/Unknown/Not Reported':0,
})
df_sex.insert(0, 'FID', 0)
df_sex.reset_index(drop=True, inplace=True)
display(df_sex)

In [ ]:
df_pcs = pd.read_csv(f'{REL11_DIR}/meta_data/qc_metrics/projected_pcs_vwb.csv')
df_pcs = df_pcs[['IID', 'PC1', 'PC2','PC3', 'PC4','PC5']].copy()
df_pcs['IID'] = df_pcs['IID'].str.replace(r'_s1$', '', regex=True)
df_pcs

In [ ]:
# Print demographic data for each ancestry
df_ages = key[["IID","AGE"]].copy()
df_ages.reset_index(drop=True, inplace=True)
display(df_ages)

In [ ]:
df_pheno = key[["IID", "phenotype"]].copy()
df_pheno.rename({"phenotype":"PHENO"}, inplace=True, axis=1)
df_pheno["PHENO"] = df_pheno["PHENO"].replace({
    "PD": 2,
    "Control": 1,
    "Other": -9,
})
display(df_pheno)

In [ ]:
df_anc = key[["IID", "nba_label"]].copy()
df_anc = df_anc.rename(columns={"nba_label": "ANCESTRY"})
df_anc

In [ ]:
df_covar = df_sex.merge(df_ages, on="IID")
df_covar

In [ ]:
df_covar = df_covar.merge(df_anc, on="IID")
df_covar

In [ ]:
df_covar = df_covar.merge(df_pheno, on="IID")
df_covar

In [ ]:
df_covar = df_covar.merge(df_pcs, on="IID")
df_covar

In [ ]:
df_covar = df_covar.dropna()
df_covar

In [ ]:
df_covar["FID"] = df_covar["IID"]
df_covar

In [ ]:
df_covar = df_covar.dropna().copy()
df_covar["FID"] = df_covar["IID"]

df_covar = df_covar[["FID","IID","SEX","AGE","PC1","PC2","PC3","PC4","PC5"]]

In [ ]:
df_covar.to_csv("/home/jupyter/workspace/ws_files/r11/results/covariate.txt", index=False, sep="\t")

In [ ]:
df_covar.to_csv("/home/jupyter/workspace/ws_files/r11/results/covariate_pheno.txt", index=False, sep="\t")

In [ ]:
# 10 ancestries loop one by one
CAH = pd.read_csv("/home/jupyter/workspace/ws_files/r11/cohort/CAH/IPD_CAH_rm.txt", sep='\t', header=None, names=['FID', 'IID'] )
CAH = CAH.dropna()
df_CAH = df_covar.merge(CAH[['IID']], on='IID', how='inner')
df_CAH

In [ ]:
all_anc = pd.concat([df_EUR, df_AAC, df_AMR, df_AJ, df_AFR, df_SAS, df_EAS, df_CAS, df_MDE, df_CAH], ignore_index=True)
all_anc

In [ ]:
all_anc["FID"] = all_anc["IID"]
all_anc

In [ ]:
IPD_samplestokeep = all_anc[["FID", "IID"]].copy()

IPD_samplestokeep.to_csv("/home/jupyter/workspace/ws_files/r11/results/IPD_samplestokeep_rm.txt", index=False, sep="\t")

In [ ]:
all_anc.to_csv("/home/jupyter/workspace/ws_files/r11/results/covariate_IPD_rm.txt", index=False, sep="\t")

In [ ]:
! awk '{print NF}' /home/jupyter/workspace/ws_files/r11/results/covariate_IPD.txt | sort -nu | head

In [ ]:
IPD = pd.read_csv("/home/jupyter/workspace/ws_files/r11/results/IPD_samplestokeep_rm.txt", sep='\t')
IPD

In [ ]:
# find rows where FID starts with "TUEPAC"
TUEPAC_IPD = IPD[IPD["FID"].str.startswith("TUEPAC", na=False)].copy()
TUEPAC_IPD

In [ ]:
# save
TUEPAC_IPD.to_csv("/home/jupyter/workspace/ws_files/r11/results/TUEPAC_IPD_samplestokeep_rm.txt", sep="\t", index=False, header=False)

In [ ]:
# exclude tuepac from IPD
IPD_no_tuepac = IPD[~(IPD["FID"].isin(tuepac["FID"]))].copy()
IPD_no_tuepac

In [ ]:
IPD_no_tuepac.to_csv("/home/jupyter/workspace/ws_files/r11/results/IPD_exclude_TUEPAC_samplestokeep_rm.txt", sep="\t", index=False, header=False)

Run PRSice

In [ ]:
%%bash
# test in "EUR_no_TUEPAC", let PRSice automatically choose best threshold

Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/EUR/EUR \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
--print-snp \
--score con-std \
--perm 10000 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--prevalence 0.005 \
--fastscore \
--no-full \
--keep /home/jupyter/workspace/ws_files/r11/results/IPD_exclude_TUEPAC_samplestokeep_rm.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/covariate.txt \
--thread 16 

In [ ]:
! head /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr1.fam

In [ ]:
! head /home/jupyter/workspace/ws_files/r11/results/covariate.txt

AUC in EUR

In [ ]:
RESULTS_DIR

In [ ]:
%%R

dat <- read.table(paste0(RESULTS_DIR, "/EUR", "/", "EUR.best"), header = TRUE, sep = " ")
cov <- read.table(paste0(RESULTS_DIR, "/covariate_pheno.txt"), header = TRUE, sep = "\t") 
colnames(cov) <- c("FID", "IID", "SEX", "AGE", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")

dat <- merge(dat, cov, by = "IID")
dat$CASE <- dat$PHENO - 1
dat <- subset(dat, CASE != -10)

meanControls <- mean(dat$PRS[dat$CASE == 0])
sdControls <- sd(dat$PRS[dat$CASE == 0])
dat$zSCORE <- (dat$PRS - meanControls) / sdControls

grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
dat$probDisease <- predict(grsTests, dat, type = "response")
dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")

roc <- roc(response = dat$reported, predictor = dat$probDisease)
auc_value <- auc(roc)

result.coords <- coords(
    roc, 
    "best", 
    best.method = "closest.topleft", 
    ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden")
)

dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

results_dt <- data.table(
  AUC               = as.numeric(auc_value),
  Accuracy          = as.numeric(confMat$overall["Accuracy"]),
  CI_Lower          = as.numeric(confMat$overall["AccuracyLower"]),
  CI_Upper          = as.numeric(confMat$overall["AccuracyUpper"]),
  Balanced_Accuracy = as.numeric(confMat$byClass["Balanced Accuracy"]),
  Sensitivity       = as.numeric(confMat$byClass["Sensitivity"]),
  Specificity       = as.numeric(confMat$byClass["Specificity"])
)

fwrite(results_dt, paste0(RESULTS_DIR, "/EUR/iPDvsHC_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

In [ ]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/EUR/iPDvsHC_results.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)

df = pd.read_csv(f"{RESULTS_DIR}/EUR/EUR.summary", sep="\s+")
df = df[[
    "Threshold",
    "PRS.R2.adj",
    "Full.R2",
    "Null.R2",
    "Coefficient",
    "Standard.Error",
    "Num_SNP",
]]
df.rename(columns={
    "PRS.R2.adj":"PRS R2 Adj",
    "Full.R2":"Full R2",
    "Null.R2":"Null R2",
    "Standard.Error":"SE",
    "Num_SNP":"No. of SNP",
}, inplace=True)
lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
df[["PRS R2 Adj","Full R2","Null R2","Coefficient","SE"]] = df[["PRS R2 Adj","Full R2","Null R2","Coefficient","SE"]].round(3)
df["Threshold"] = df["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = pd.concat([df_bestfit, df], axis=1)
df_merged.to_csv(f"{RESULTS_DIR}/EUR/table3.txt", index=False, sep="\t")

Plot ROC

In [ ]:
%%R

prs_data <- fread(paste0(RESULTS_DIR, '/EUR/', 'EUR.best'), header=T)
colnames(prs_data)[4] <- "PRS"
covar <- fread(paste0(RESULTS_DIR, "/covariate_pheno.txt"), header=T)
colnames(covar)[1] <- "FID"
colnames(covar)[2] <- "IID"
temp <- merge(prs_data, covar, by = c("FID","IID"))
temp$CASE <- temp$PHENO - 1
DATA <- subset(temp, CASE != -10)

## Probability of disease calculation
Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
DATA$probDisease <- predict(Model, DATA, type = c("response"))
DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

write.table(DATA, file = paste0(RESULTS_DIR, "/EUR/risk_results_r11.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
to_plot = pd.read_csv(f"{RESULTS_DIR}/EUR/risk_results_r11.txt", sep="\t")
plt.figure(figsize=(8, 5))

fpr, tpr, _ = roc_curve(to_plot["CASE"], to_plot["probDisease"])
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'(AUC = {roc_auc:.2f})')

sns.set_palette("Dark2")
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title("ROC Curves- iPD")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.savefig(f"{RESULTS_DIR}/EUR/figure4_r11.png", dpi=300)
plt.close()

calculate p value including covariates in regression

In [ ]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/EUR/risk_results_r11.txt")
head(data)

In [ ]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/EUR/risk_results_r11.csv")

In [ ]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

In [ ]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [ ]:
%%R

model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5,
               family = "binomial", data = data)

summary(model)

coef_tab <- as.data.frame(coef(summary(model)))

# use a safer digits, e.g. 16 (usually max for double precision printing)
coef_tab$P_pretty <- format.pval(coef_tab[["Pr(>|z|)"]], digits = 16, eps = 0)

print(coef_tab)

In [ ]:
%%R

capture.output(
  coef_tab,
  file = paste0(RESULTS_DIR, "/EUR/EUR_model_summary.txt")
)

In [ ]:
%%bash
# test in "AAC","AJ","CAS","MDE","CAH"

Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/CAH/CAH \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/CAH/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/r11/genotype_file/CAH/chr# \
--print-snp \
--score con-std \
--perm 10000 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--prevalence 0.005 \
--fastscore \
--no-full \
--keep /home/jupyter/workspace/ws_files/r11/results/IPD_exclude_TUEPAC_samplestokeep_rm.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/covariate.txt \
--thread 16 

In [ ]:
%%bash
# test in "AFR","AMR","SAS","EUR","EAS"

Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/EAS/EAS \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EAS/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EAS \
--print-snp \
--score con-std \
--perm 10000 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--prevalence 0.005 \
--fastscore \
--no-full \
--keep /home/jupyter/workspace/ws_files/r11/results/IPD_exclude_TUEPAC_samplestokeep_rm.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/covariate.txt \
--thread 16 

In [ ]:
%%bash
# run this on terminal

for ancestry in {"AFR","AMR","SAS","EUR","EAS"}; do
    Rscript /home/jupyter/PRSice.R \
    --out /home/jupyter/workspace/ws_files/results/${ancestry}/${ancestry}_updated \
    --target /home/jupyter/workspace/ws_files/results/${ancestry}/target_file/chr# \
    -b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
    --beta \
    --snp SNP \
    --a1 A1 \
    --a2 A2 \
    --stat BETA \
    --pvalue P \
    --ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_${ancestry} \
    --print-snp \
    --score con-std \
    --perm 10000 \
    --prsice /home/jupyter/prsice \
    -n 24 \
    --binary-target T \
    --quantile 4 \
    --prevalence 0.005 \
    --fastscore \
    --no-full \
    --thread 16 \
    --keep /home/jupyter/workspace/ws_files/results/covariate_IPD.txt \
    --cov-file /home/jupyter/workspace/ws_files/results/covariate.txt
done

In [ ]:
%%bash
# run this on terminal

for ancestry in {"AAC","AJ","CAS","MDE","CAH"}; do
    Rscript /home/jupyter/PRSice.R \
    --out /home/jupyter/workspace/ws_files/results/${ancestry}/${ancestry}_updated \
    --target /home/jupyter/workspace/ws_files/results/${ancestry}/target_file/chr# \
    -b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
    --beta \
    --snp SNP \
    --a1 A1 \
    --a2 A2 \
    --stat BETA \
    --pvalue P \
    --ld /home/jupyter/workspace/ws_files/results/${ancestry}/target_file/chr# \
    --print-snp \
    --score con-std \
    --perm 10000 \
    --prsice /home/jupyter/prsice \
    -n 24 \
    --binary-target T \
    --quantile 4 \
    --prevalence 0.005 \
    --fastscore \
    --no-full \
    --thread 16 \
    --keep /home/jupyter/workspace/ws_files/results/covariate_IPD.txt \
    --cov-file /home/jupyter/workspace/ws_files/results/covariate.txt
done

estimate specificity and sensitivity

In [ ]:
RESULTS_DIR

In [ ]:
ancestries = ['AMR', 'AAC', 'AJ', 'SAS', 'AFR', 'EUR', 'EAS', 'CAS', 'MDE', 'CAH']

In [ ]:
! head /home/jupyter/workspace/ws_files/r11/results/covariate_IPD_rm.txt

In [ ]:
%%R

# Initialize results data.table
results_dt <- data.table(Ancestry = ancestries)
results_dt[, AUC := as.numeric(NA)]
results_dt[, Accuracy := as.numeric(NA)]
results_dt[, CI_Lower := as.numeric(NA)]
results_dt[, CI_Upper := as.numeric(NA)]
results_dt[, Balanced_Accuracy := as.numeric(NA)]
results_dt[, Sensitivity := as.numeric(NA)]
results_dt[, Specificity := as.numeric(NA)]

for (ancestry in ancestries) {
    print(ancestry)

    dat <- read.table(paste0(RESULTS_DIR, "/", ancestry, "/", ancestry, ".best"), header = TRUE, sep = " ")
    cov <- read.table(paste0(RESULTS_DIR, "/covariate_IPD_rm.txt"), header = TRUE, sep = "\t") 
    colnames(cov) <- c("FID", "IID", "SEX", "AGE", "ANCESTRY", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")
    dat <- merge(dat, cov, by = "IID")
    dat$CASE <- dat$PHENO - 1
    dat <- subset(dat, CASE != -10)
    meanControls <- mean(dat$PRS[dat$CASE == 0])
    sdControls <- sd(dat$PRS[dat$CASE == 0])
    dat$zSCORE <- (dat$PRS - meanControls) / sdControls
    grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
    dat$probDisease <- predict(grsTests, dat, type = "response")
    dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")
    roc <- roc(response = dat$reported, predictor = dat$probDisease)
    auc_value <- auc(roc)

    result.coords <- coords(
        roc, 
        "best", 
        best.method = "closest.topleft", 
        ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden"),
    )

    dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
    confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

    results_dt[Ancestry == ancestry, AUC := auc_value]
    results_dt[Ancestry == ancestry, Accuracy := confMat$overall["Accuracy"]]
    results_dt[Ancestry == ancestry, CI_Lower := confMat$overall["AccuracyLower"]]
    results_dt[Ancestry == ancestry, CI_Upper := confMat$overall["AccuracyUpper"]]
    results_dt[Ancestry == ancestry, Balanced_Accuracy := confMat$byClass["Balanced Accuracy"]]
    results_dt[Ancestry == ancestry, Sensitivity := confMat$byClass["Sensitivity"]]
    results_dt[Ancestry == ancestry, Specificity := confMat$byClass["Specificity"]]
}

fwrite(results_dt, paste0(RESULTS_DIR, "/IPD/r11_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

In [ ]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/IPD/r11_results.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)

df_prsice = []
for ancestry in ancestries:
    df = pd.read_csv(f"{RESULTS_DIR}/{ancestry}/{ancestry}.summary", sep="\s+")
    df["Ancestry"] = ancestry
    df = df[[
        "Ancestry",
        "Threshold",
        "PRS.R2.adj",
        "Full.R2",
        "Null.R2",
        "Coefficient",
        "Standard.Error",
        "Num_SNP",
    ]]
    df.rename(columns={
        "PRS.R2.adj":"PRS R2 Adj",
        "Full.R2":"Full R2",
        "Null.R2":"Null R2",
        "Standard.Error":"SE",
        "Num_SNP":"No. of SNP",
    }, inplace=True)
    lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
    upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
    df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
    df_prsice.append(df)
df_prsice = pd.concat(df_prsice)
df_prsice[["PRS R2 Adj","Full R2","Null R2","Coefficient","SE"]] = df_prsice[["PRS R2 Adj","Full R2","Null R2","Coefficient","SE"]].round(3)
df_prsice["Threshold"] = df_prsice["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = df_prsice.merge(df_bestfit, on="Ancestry")
df_merged.to_csv(f"{RESULTS_DIR}/IPD/table3_r11.txt", index=False, sep="\t")

Plot ROC

In [ ]:
%%R

data_list <- list()
i <- 1
for (ancestry in ancestries) {
    print(paste0("------------------------------", ancestry, "------------------------------"))

    prs_data <- fread(paste0(RESULTS_DIR, '/', ancestry,'/', ancestry, '.best'), header=T)
    colnames(prs_data)[4] <- "PRS"
    covar <- fread(paste0(RESULTS_DIR, "/covariate_IPD_rm.txt"), header=T)
    colnames(covar)[1] <- "FID"
    colnames(covar)[2] <- "IID"
    temp <- merge(prs_data, covar, by = c("FID","IID"))
    temp$CASE <- temp$PHENO - 1
    DATA <- subset(temp, CASE != -10)

    ## Probability of disease calculation
    Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
    DATA$probDisease <- predict(Model, DATA, type = c("response"))
    DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
    DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

    ## Probability of disease calculation
    Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
    DATA$probDisease <- predict(Model, DATA, type = c("response"))
    DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
    DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

    dat <- DATA
    dat$Group.name <- paste0(ancestry)
    # dat$Group <- "Model 2"
    data_list[[i]] <- dat
    i <- i + 1
}

to_plot <- rbindlist(data_list, fill = TRUE)
write.table(to_plot, file = paste0(RESULTS_DIR, "/IPD/risk_results_r11.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
to_plot = pd.read_csv(f"{RESULTS_DIR}/IPD/risk_results_r11.txt", sep="\t")
plt.figure(figsize=(8, 5))
ancestry_groups = to_plot['Group.name'].unique()
for ancestry_group in ancestry_groups:
    group_data = to_plot[to_plot['Group.name'] == ancestry_group]
    fpr, tpr, _ = roc_curve(group_data["CASE"], group_data["probDisease"])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{ancestry_group} (AUC = {roc_auc:.2f})')

sns.set_palette("Dark2")
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title("ROC Curves by Ancetry")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.savefig(f"{RESULTS_DIR}/IPD/AUC_r11.png", dpi=300)
plt.close()

In [ ]:
to_plot = pd.read_csv(f"{RESULTS_DIR}/IPD/risk_results_r11.txt", sep="\t")
plt.figure(figsize=(10, 8))
ancestry_groups = to_plot['Group.name'].unique()

# --- compute all ROCs first, then sort by AUC (best first) ---
roc_results = []
for ancestry_group in ancestry_groups:
    group_data = to_plot[to_plot['Group.name'] == ancestry_group]
    fpr, tpr, _ = roc_curve(group_data["CASE"], group_data["probDisease"])
    roc_auc = auc(fpr, tpr)
    roc_results.append((ancestry_group, fpr, tpr, roc_auc))
roc_results.sort(key=lambda x: x[3], reverse=True)

# --- one distinct color per ancestry, assigned best -> worst ---
n = len(roc_results)
colors = plt.cm.tab10.colors                       # 10 distinct colors
widths = [2.8 - 1.2 * i / (n - 1) for i in range(n)]   # thick -> thin by rank

for i, (ancestry_group, fpr, tpr, roc_auc) in enumerate(roc_results):
    plt.plot(fpr, tpr, color=colors[i % len(colors)], linewidth=widths[i], linestyle='-',
             label=f'{ancestry_group} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], color='0.6', linestyle=':', linewidth=1)
plt.title("ROC Curves by Ancetry")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right", fontsize=9, labelspacing=0.8, framealpha=1.0)
plt.savefig(f"{RESULTS_DIR}/IPD/AUC_r11_01.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
to_plot = pd.read_csv(f"{RESULTS_DIR}/IPD/risk_results_r11.txt", sep="\t")
plt.figure(figsize=(8, 5))
ancestry_groups = to_plot['Group.name'].unique()

roc_results = []
for ancestry_group in ancestry_groups:
    group_data = to_plot[to_plot['Group.name'] == ancestry_group]
    fpr, tpr, _ = roc_curve(group_data["CASE"], group_data["probDisease"])
    roc_auc = auc(fpr, tpr)
    roc_results.append((ancestry_group, fpr, tpr, roc_auc))
roc_results.sort(key=lambda x: x[3], reverse=True)

for ancestry_group, fpr, tpr, roc_auc in roc_results:
    plt.plot(fpr, tpr, label=f'{ancestry_group} (AUC = {roc_auc:.2f})')

sns.set_palette("Dark2")
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title("ROC Curves by Ancetry")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.savefig(f"{RESULTS_DIR}/IPD/AUC_r11_02.png", dpi=300)
plt.close()

violin plot

In [ ]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/IPD/risk_results_r11.txt")
head(data)

In [ ]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/IPD/risk_results_r11.csv")

In [ ]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

In [ ]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [ ]:
%%R

ancestries <- unique(data$ANCESTRY)
logit_results <- list()

for (ancestry in ancestries) {
  df <- subset(data, ANCESTRY == ancestry)

  model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5,
               family = "binomial", data = df)

  sm <- coef(summary(model)) |> as.data.frame()
  sm$Variable <- rownames(sm)

  z <- abs(sm$`z value`)
  logp <- pnorm(z, lower.tail = FALSE, log.p = TRUE) + log(2)  # ln(p)
  exp10 <- floor(logp / log(10))
  mant  <- 10^((logp / log(10)) - exp10)
  sm$p_pretty <- sprintf("%.4fE%+d", mant, as.integer(exp10))

  sm$Ancestry <- ancestry   # keep ancestry label
  rownames(sm) <- NULL

  logit_results[[ancestry]] <- sm

  cat("\n===== Ancestry:", ancestry, "=====\n")
  print(sm[, c("Ancestry","Variable","Estimate","Std. Error","z value","Pr(>|z|)","p_pretty")],
        row.names = FALSE)
}

# Combine all results
final_results <- do.call(rbind, logit_results)

# Save as TXT (tab-separated)
write.table(final_results, file = "/home/jupyter/workspace/ws_files/r11/results/IPD/logistic_results.txt",
            sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
%%R

ancestries <- unique(data$ANCESTRY)

logit_results <- list()

for (ancestry in ancestries) {
  df <- subset(data, ANCESTRY == ancestry)

  # Logistic regression
  model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5, family = "binomial", data = df)

  summary_model <- coef(summary(model))
  summary_model <- as.data.frame(summary_model)
  summary_model$Variable <- rownames(summary_model)
  rownames(summary_model) <- NULL
  logit_results[[ancestry]] <- summary_model
  cat("\n===== Ancestry:", ancestry, "=====\n")
  print(summary_model)  # Shows p-values, std error, etc.
}